In [1]:
import cv2
import mediapipe as mp
from keras.models import load_model
import numpy as np

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.5)
mp_draw = mp.solutions.drawing_utils


In [2]:
model = load_model("E:\\ML\\handsign\\models\\model_handsign4.h5") 
danh_sach_chu = ["a", "b", "c", "d", "e", "f", "g", "h", "i", "k", "l", "m", "n", "o", "p", "q", "r", "s", "t", "u", "v", "w", "x", "y"]


In [3]:
def khoang_cach_toa_do(hand_lm):
    danh_sach_toa_do = []
    for lm in hand_lm.landmark:
        danh_sach_toa_do.append([lm.x, lm.y, lm.z])
    toa_do = np.array(danh_sach_toa_do)
    co_tay = toa_do[0]
    chieu_dai = np.linalg.norm(toa_do[9] - co_tay) + 1e-6
    chuan_hoa = (toa_do - co_tay) / chieu_dai

    ngon_tay = [4, 8, 12, 16, 20]
    khoang_cach = []
    for i in ngon_tay:
        khoang_cach.append(np.linalg.norm((toa_do[i] - co_tay) / chieu_dai))
    for i in range(len(ngon_tay)-1):
        khoang_cach.append(np.linalg.norm(toa_do[ngon_tay[i]] - toa_do[ngon_tay[i+1]]) / chieu_dai)

    return np.hstack([chuan_hoa.flatten(), khoang_cach])


In [4]:
def chong_nhieu(du_doan, khung_hinh, danh_sach_chu, so_frame_xet=20):
    
    frame_hien_tai = np.argmax(du_doan)                       # Kết quả dự đoán hiện tại
    
    if len(khung_hinh) == 0:                                  # Nếu khung_hinh chưa có giá trị thì thêm frame hiện tại vào
        khung_hinh.append(frame_hien_tai)
    elif len(khung_hinh) < so_frame_xet:                      # Khi chưa đủ số lượng frame xét
        if frame_hien_tai != khung_hinh[-1]:                  # Nếu frame hiện tại khác frame trước thì xét lại từ đầu
            khung_hinh.clear()
            khung_hinh.append(frame_hien_tai)                 # Thêm lại frame hiện tại làm khởi đầu mới
        else:
            khung_hinh.append(frame_hien_tai)                 # Nếu frame hiện tại giống frame trước thì thêm vào khung_hinh
            
    if len(khung_hinh) == so_frame_xet:                       # Nếu đạt đủ số frames giống nhau liên tiếp
        chu_du_doan = danh_sach_chu[khung_hinh[-1]]         
        khung_hinh.clear()
        return chu_du_doan
        
    return None

In [ ]:
cam = cv2.VideoCapture(0)
khung_hinh = [] # Khởi tạo list trống để lưu lịch sử frame

while True:
    ret, frame = cam.read()    
    if not ret: break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)    
    h_frames, w_frames, c = frame.shape

    if results.multi_hand_landmarks:
        for hand_lm in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_lm, mp_hands.HAND_CONNECTIONS)
            
            du_lieu = np.array([khoang_cach_toa_do(hand_lm)])
            du_doan = model.predict(du_lieu, verbose=0)   
            phan_tram = np.max(du_doan) 
            ket_qua_chu = danh_sach_chu[np.argmax(du_doan)]
            
            if phan_tram > 0.6:
                x_co_tay = int(hand_lm.landmark[0].x * w_frames)
                y_co_tay = int(hand_lm.landmark[0].y * h_frames)
                cv2.putText(frame, f"{ket_qua_chu.upper()}: {phan_tram*100:.1f}%", 
                            (x_co_tay - 20, y_co_tay - 20), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            ket_qua_cn = chong_nhieu(du_doan, khung_hinh, danh_sach_chu, so_frame_xet=20)
            if ket_qua_cn is not None:
                print(f"ket qua: {ket_qua_cn}")

    cv2.imshow("Test Handsign", frame)
    if cv2.waitKey(1) & 0xFF == ord('l'): break

cam.release()
cv2.destroyAllWindows()


Chữ nhận diện ổn định: b
Chữ nhận diện ổn định: b
Chữ nhận diện ổn định: b
Chữ nhận diện ổn định: b
Chữ nhận diện ổn định: b
